# 연습용 노트북 — Python / Pandas / yfinance 기초

> 이 노트북은 프로젝트에 직접 쓰이지 않는 **코드 연습 공간**이다.  
> 자유롭게 실험하고 망가뜨려도 된다.

---

## 목차
1. pandas 기초 — 데이터프레임 만들기
2. yfinance로 주가 가져오기
3. 수익률 계산
4. matplotlib으로 차트 그리기
5. 섹터별 비교 연습

---
## 0. 패키지 임포트

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 한글 폰트 (macOS)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

print('패키지 임포트 완료')

---
## 1. pandas 기초 — 데이터프레임 만들기

프로젝트에서 `data/reference/companies.csv`처럼 생긴 데이터를 직접 만들어보는 연습.

In [ ]:
# 딕셔너리 → 데이터프레임
data = {
    'ticker':  ['NVDA', 'AMD',  'VRT',  'ETN',  'NEE'],
    'company': ['NVIDIA', 'AMD', 'Vertiv', 'Eaton', 'NextEra'],
    'sector':  ['semiconductor', 'semiconductor', 'cooling', 'power_equipment', 'power_utility'],
    'bottleneck_relevance': ['high', 'mid', 'high', 'high', 'mid'],
}

df_companies = pd.DataFrame(data)
df_companies

In [ ]:
# 기본 조작 연습
print('행 수:', len(df_companies))
print('컬럼:', df_companies.columns.tolist())
print()

# 필터링 — 섹터 기준
df_semi = df_companies[df_companies['sector'] == 'semiconductor']
print('반도체 기업:')
print(df_semi[['ticker', 'company']])

In [ ]:
# 섹터별 기업 수 세기
df_companies.groupby('sector')['ticker'].count().sort_values(ascending=False)

---
## 2. yfinance로 주가 가져오기

실제 프로젝트의 `src/data/fetch_prices.py`가 하는 일을 직접 해보는 연습.

In [ ]:
def hello():
    print("안녕")

a = hello

print(hello)
print(a)

hello()
a()


안녕


In [ ]:
def add(x, y):
    return x + y

b = add

print(b(3, 5))

In [15]:
import asyncio

async def work(name):
    print(f"{name} 시작")
    await asyncio.sleep(1)
    print(f"{name} 끝")

async def main():
    await asyncio.gather(
        work("A"),
        work("B"),
        work("C")
    )

await main()

A 시작
B 시작
C 시작
A 끝
B 끝
C 끝


In [11]:
import asyncio
async def main():
    print("비동기 함수 실행")
    await asyncio.sleep(1)
    print("\n1초 후 실행완료")

main()
await main()

/var/folders/vf/9p8syxcx1g77s8jzp8b8b8fh0000gn/T/ipykernel_21686/2045218476.py:7: RuntimeWarning: coroutine 'main' was never awaited
  main()


비동기 함수 실행

1초 후 실행완료


In [16]:
print(dir(asyncio))

['ALL_COMPLETED', 'AbstractChildWatcher', 'AbstractEventLoop', 'AbstractEventLoopPolicy', 'AbstractServer', 'BaseEventLoop', 'BaseProtocol', 'BaseTransport', 'BoundedSemaphore', 'BufferedProtocol', 'CancelledError', 'Condition', 'DatagramProtocol', 'DatagramTransport', 'DefaultEventLoopPolicy', 'Event', 'FIRST_COMPLETED', 'FIRST_EXCEPTION', 'FastChildWatcher', 'Future', 'Handle', 'IncompleteReadError', 'InvalidStateError', 'LifoQueue', 'LimitOverrunError', 'Lock', 'MultiLoopChildWatcher', 'PidfdChildWatcher', 'PriorityQueue', 'Protocol', 'Queue', 'QueueEmpty', 'QueueFull', 'ReadTransport', 'SafeChildWatcher', 'SelectorEventLoop', 'Semaphore', 'SendfileNotAvailableError', 'Server', 'StreamReader', 'StreamReaderProtocol', 'StreamWriter', 'SubprocessProtocol', 'SubprocessTransport', 'Task', 'ThreadedChildWatcher', 'TimeoutError', 'TimerHandle', 'Transport', 'WriteTransport', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__'

In [18]:
import inspect
import asyncio

print(inspect.getsource(asyncio.sleep))

async def sleep(delay, result=None):
    """Coroutine that completes after a given time (in seconds)."""
    if delay <= 0:
        await __sleep0()
        return result

    loop = events.get_running_loop()
    future = loop.create_future()
    h = loop.call_later(delay,
                        futures._set_result_unless_cancelled,
                        future, result)
    try:
        return await future
    finally:
        h.cancel()



In [21]:
import asyncio

async def work(n):
    print(f"{n} 시작")
    await asyncio.sleep(1)
    print(f"{n} 끝")

async def main():
    await asyncio.gather(
        work(1),
        work(2),
        work(3)
    )

await main()

1 시작
2 시작
3 시작
1 끝
2 끝
3 끝


In [24]:
import asyncio

async def work(n):
    print(f"{n} 시작")
    await asyncio.sleep(1)
    print(f"{n} 끝")

async def main():
    task=asyncio.create_task(
        work(1)
       )
    await task
await main()

1 시작
1 끝


In [ ]:
import asyncio

async def work():
    print("작업 시작")
    await asyncio.sleep(2)
    print("작업 끝")
    return "결과값"

async def main():
    task = asyncio.create_task(work()) # 지금 바로 결과 받지 않을게 일은 먼저 시작해둬"

    print("나는 다른 일 1")
    await asyncio.sleep(1)
    print("나는 다른 일 2")

    result = await task # 이제 결과줘
    print("저장된 결과:", result)

await main()

나는 다른 일 1
작업 시작
나는 다른 일 2
작업 끝
저장된 결과: 결과값


In [32]:
from mcp.server import Server
print(dir(Server))

['__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_get_cached_tool_definition', '_handle_message', '_handle_notification', '_handle_request', '_is_protocol', '_make_error_result', 'call_tool', 'completion', 'create_initialization_options', 'experimental', 'get_capabilities', 'get_prompt', 'list_prompts', 'list_resource_templates', 'list_resources', 'list_tools', 'progress_notification', 'read_resource', 'request_context', 'run', 'set_logging_level', 'subscribe_resource', 'unsubscribe_resource']


In [34]:
# data_collector_mcp/server.py

from mcp.server import Server
from mcp.server.stdio import stdio_server
import yfinance as yf

# ① 서버 인스턴스 생성
app = Server("data-collector")

# ② 함수(tool) 등록
@app.tool()
async def get_stock_price(ticker: str) -> str:
    """주가를 가져오는 함수"""
    
    stock = yf.Ticker(ticker)        # yfinance로 데이터 요청
    price = stock.info["currentPrice"]
    return f"{ticker} 현재가: ${price}"

@app.tool()
async def get_financial_data(ticker: str) -> str:
    """재무제표를 가져오는 함수"""
    
    stock = yf.Ticker(ticker)
    income = stock.financials          # 손익계산서
    revenue = income.loc["Total Revenue"].iloc[0]
    return f"{ticker} 최근 매출: ${revenue:,.0f}"

# ③ 서버 실행
async def main():
    async with stdio_server() as streams:
        await app.run(*streams)

AttributeError: 'Server' object has no attribute 'tool'

In [ ]:
# 티커 하나 — 기본 정보 확인
ticker = yf.Ticker('NVDA')

# 기본 정보
info = ticker.info
print('회사명:', info.get('longName'))
print('시가총액:', f"{info.get('marketCap', 0) / 1e12:.2f}조 달러")
print('52주 고가:', info.get('fiftyTwoWeekHigh'))
print('52주 저가:', info.get('fiftyTwoWeekLow'))

In [ ]:
# 여러 티커 — 주가 일별 데이터 다운로드
TICKERS = ['NVDA', 'AMD', 'VRT', 'ETN', 'NEE', 'SPY']  # SPY = S&P 500 벤치마크

df_raw = yf.download(TICKERS, period='6mo', auto_adjust=True)
print('데이터 shape:', df_raw.shape)
df_raw['Close'].tail()

In [ ]:
# 종가만 추출 (분석에서 주로 쓰는 형태)
df_prices = df_raw['Close'].copy()
df_prices.index = pd.to_datetime(df_prices.index)

print('결측치 확인:')
print(df_prices.isnull().sum())

# 결측치 처리 — 앞 값으로 채우기
df_prices = df_prices.ffill()
df_prices.tail(3)

---
## 3. 수익률 계산

실제 프로젝트의 `src/analysis/returns.py`가 하는 일.

In [ ]:
def calc_return(df: pd.DataFrame, days: int) -> pd.Series:
    """n 거래일 기준 수익률 계산. 음수면 손실."""
    return (df.iloc[-1] / df.iloc[-days] - 1) * 100

# 기간별 수익률
df_returns = pd.DataFrame({
    'ret_1w':  calc_return(df_prices, 5),
    'ret_1m':  calc_return(df_prices, 21),
    'ret_3m':  calc_return(df_prices, 63),
}).round(2)

df_returns

In [ ]:
# SPY 대비 상대 수익률 (rel_to_spy)
spy_3m = df_returns.loc['SPY', 'ret_3m']

df_returns['rel_to_spy_3m'] = (df_returns['ret_3m'] - spy_3m).round(2)

# SPY 제외하고 정렬
df_returns.drop('SPY').sort_values('rel_to_spy_3m', ascending=False)

In [ ]:
# 간단한 병목 점수 (v1) — 섹터 평균 대비
# 프로젝트 CLAUDE.md: bottleneck_score_v1 = 섹터_3M_수익률 - 전체_AI인프라_평균_3M_수익률
ai_infra_tickers = ['NVDA', 'AMD', 'VRT', 'ETN', 'NEE']
avg_3m = df_returns.loc[ai_infra_tickers, 'ret_3m'].mean()

df_returns['bottleneck_score_v1'] = (df_returns['ret_3m'] - avg_3m).round(2)

print(f'AI 인프라 평균 3M 수익률: {avg_3m:.2f}%')
df_returns.loc[ai_infra_tickers, ['ret_3m', 'bottleneck_score_v1']].sort_values('bottleneck_score_v1', ascending=False)

---
## 4. matplotlib으로 차트 그리기

In [ ]:
# 정규화 주가 차트 (시작점 = 100)
df_norm = df_prices[['NVDA', 'AMD', 'VRT', 'ETN', 'NEE', 'SPY']] / df_prices[['NVDA', 'AMD', 'VRT', 'ETN', 'NEE', 'SPY']].iloc[0] * 100

fig, ax = plt.subplots(figsize=(12, 6))

colors = {'NVDA': '#76b900', 'AMD': '#ed1c24', 'VRT': '#0072c6',
          'ETN': '#ff6900', 'NEE': '#009150', 'SPY': 'gray'}

for ticker in df_norm.columns:
    ls = '--' if ticker == 'SPY' else '-'
    lw = 1.5 if ticker == 'SPY' else 2
    ax.plot(df_norm.index, df_norm[ticker], label=ticker, color=colors[ticker], linestyle=ls, linewidth=lw)

ax.axhline(100, color='black', linewidth=0.5, linestyle=':')
ax.set_title('AI 인프라 주요 기업 — 6개월 정규화 주가 (시작=100)', fontsize=14)
ax.set_ylabel('지수 (시작=100)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 3M 수익률 막대 그래프
df_plot = df_returns.loc[ai_infra_tickers, 'ret_3m'].sort_values(ascending=True)

colors_bar = ['#d32f2f' if v < 0 else '#1976d2' for v in df_plot.values]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(df_plot.index, df_plot.values, color=colors_bar)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('3개월 수익률 (%)')
ax.set_title('AI 인프라 섹터 — 3개월 수익률 비교')

for bar, val in zip(bars, df_plot.values):
    ax.text(val + (0.5 if val >= 0 else -0.5), bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.show()

---
## 5. 자유 실험 공간

아래 셀들은 비어있다. 자유롭게 써도 된다.

In [ ]:
# 여기에 자유롭게 실험

In [ ]:
# 아이디어: 변동성(표준편차) 계산해보기
# df_prices.pct_change().std() * (252 ** 0.5) * 100  # 연환산 변동성 (%)

In [ ]:
# 아이디어: 상관관계 히트맵
# import seaborn as sns
# corr = df_prices.pct_change().corr()
# sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')